In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Set working directory (dataset directory)
data_dir = '/content/drive/MyDrive/Fed_auth'
file_path = "/content/drive/MyDrive/Fed_auth/f9cc94d7-a74f-4747-999f-04d4dca1c27c/camera"

# **Load images and create the dataset**

**center crop**

In [ ]:
import os
import random
from collections import defaultdict

import torch
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader, Subset

# -----------------------
# Paths & categories
# -----------------------
dataset_dir = "/content/drive/MyDrive/Fed_auth"

face_categories = {
    "normal", "eyes", "glasses", "mask", "mask_glasses",
    "hat", "mask_hat_glasses", "left_profile", "right_profile"
}

def is_valid_file(path):
    """Accept only camera images whose filename contains any of the face categories."""
    path_norm = path.replace("\\", "/").lower()
    fname = os.path.basename(path_norm)
    ext_ok = fname.endswith((".png", ".jpg", ".jpeg"))
    in_camera = "/camera/" in path_norm  # only use camera modality
    cat_ok = any(cat in fname for cat in face_categories)
    return ext_ok and in_camera and cat_ok

# -----------------------
# Transforms
# -----------------------
# MobileNetV2 expects 224x224, ImageNet normalization
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),  # replace with face-aligned crop later, in case of integrate MTCNN/RetinaFace
    # the model sees slightly different versions of the same image across epochs of the training process, with the following lines:
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(p=0.5),
    # (Optional) slight blur/noise to regularize
    # transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

transform_eval = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

# -----------------------
# Dataset views
# -----------------------
# Two views over the same file set, different transforms
dataset_train_view = ImageFolder(
    root=dataset_dir,
    transform=transform_train,
    is_valid_file=is_valid_file
)
dataset_eval_view = ImageFolder(
    root=dataset_dir,
    transform=transform_eval,
    is_valid_file=is_valid_file
)

# -----------------------
# Split choice
# -----------------------
# Set this to True for open-set split (train/test subjects disjoint).
OPEN_SET_BY_CLASS = False
SEED = 42
TEST_RATIO = 0.2  # 70% train, 30% test by default

targets = dataset_train_view.targets  # identical to eval_view
rng = random.Random(SEED)

if not OPEN_SET_BY_CLASS:
    # -------- Closed-set: stratified image-level split per subject --------
    indices_by_class = defaultdict(list)
    for idx, y in enumerate(targets):
        indices_by_class[y].append(idx)

    train_indices, test_indices = [], []
    for y, idxs in indices_by_class.items():
        rng.shuffle(idxs)
        n = len(idxs)
        if n == 1:
            # If a class has only 1 image, keep it in train to avoid empty test per class
            train_indices.extend(idxs)
            continue
        n_test = max(1, int(round(TEST_RATIO * n)))
        n_train = n - n_test
        # Guard rails to keep both splits non-empty
        if n_train == 0:
            n_train, n_test = 1, n - 1
        train_indices.extend(idxs[:n_train])
        test_indices.extend(idxs[n_train:])

else:
    # -------- Open-set: split at the class (subject) level --------
    all_classes = list(range(len(dataset_train_view.classes)))
    rng.shuffle(all_classes)
    n_cls = len(all_classes)
    n_test_cls = max(1, int(round(TEST_RATIO * n_cls)))
    test_cls = set(all_classes[:n_test_cls])
    train_cls = set(all_classes[n_test_cls:])

    train_indices = [i for i, y in enumerate(targets) if y in train_cls]
    test_indices  = [i for i, y in enumerate(targets) if y in test_cls]

# -----------------------
# Subsets & loaders
# -----------------------
train_set = Subset(dataset_train_view, train_indices)
test_set  = Subset(dataset_eval_view,  test_indices)

train_loader = DataLoader(train_set, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f"Subjects (classes): {len(dataset_train_view.classes)}")
print(f"Train/Test sizes: {len(train_set)}, {len(test_set)}")
print(f"Open-set by class: {OPEN_SET_BY_CLASS}")

Subjects (classes): 33
Train/Test sizes: 1441, 368
Open-set by class: False


**Face-aligned crop**

In [ ]:
import os
import random
from collections import defaultdict

import torch
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader, Subset


# -----------------------
# Paths & categories
# -----------------------
dataset_dir = "/content/drive/MyDrive/Fed_auth"   # <-- update if needed

face_categories = {
    "normal", "eyes", "glasses", "mask", "mask_glasses",
    "hat", "mask_hat_glasses", "left_profile", "right_profile"
}

def is_valid_file(path: str) -> bool:
    """Accept only camera images whose filename contains any of the face categories."""
    path_norm = path.replace("\\", "/").lower()
    fname = os.path.basename(path_norm)
    ext_ok = fname.endswith((".png", ".jpg", ".jpeg"))
    in_camera = "/camera/" in path_norm  # only use camera modality
    cat_ok = any(cat in fname for cat in face_categories)
    return ext_ok and in_camera and cat_ok


# -----------------------
# Transforms (MobileNetV2 expects 224x224 + ImageNet normalization)
# -----------------------
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

# Try to import MTCNN (facenet-pytorch). If not available, we auto-fallback.
try:
    from facenet_pytorch import MTCNN
    _HAS_MTCNN = True
except Exception:
    _HAS_MTCNN = False


class FaceAlignCrop(torch.nn.Module):
    """
    Returns a face-aligned 224x224 crop using MTCNN.
    If detection/alignment fails or MTCNN isn't available, falls back to CenterCrop(224).

    Inputs: PIL.Image
    Output: PIL.Image (224x224)
    """
    def __init__(self, image_size=224, device=None, fallback=None, margin=0, keep_all=False):
        super().__init__()
        self.image_size = image_size
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.fallback = fallback or transforms.CenterCrop(image_size)

        if _HAS_MTCNN:
            self.mtcnn = MTCNN(
                image_size=image_size,  # final aligned size
                margin=margin,          # extra margin around the face
                post_process=True,      # converts to float [0,1] + standard alignment
                keep_all=keep_all,      # False -> best face
                device=self.device
            )
        else:
            self.mtcnn = None

        self.to_pil = transforms.ToPILImage()

    def forward(self, img):
        # If MTCNN is unavailable, fallback
        if self.mtcnn is None:
            return self.fallback(img)

        # Run detection + alignment
        aligned_tensor = self.mtcnn(img)  # returns CxHxW tensor or None
        if aligned_tensor is None:
            return self.fallback(img)

        # Convert aligned tensor back to PIL for downstream augmentations
        aligned_pil = self.to_pil(aligned_tensor)
        return aligned_pil


# Compose training transform (face-aligned, then augmentations)
face_align_train = FaceAlignCrop(
    image_size=224,
    device=("cuda" if torch.cuda.is_available() else "cpu"),
    fallback=transforms.CenterCrop(224),
    margin=0,
    keep_all=False
)

transform_train = transforms.Compose([
    transforms.Resize(256),           # manageable scale before alignment
    face_align_train,                 # face-aligned crop to 224
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

# Compose evaluation transform (deterministic, no heavy augmentation)
face_align_eval = FaceAlignCrop(
    image_size=224,
    device=("cuda" if torch.cuda.is_available() else "cpu"),
    fallback=transforms.CenterCrop(224),
    margin=0,
    keep_all=False
)

transform_eval = transforms.Compose([
    transforms.Resize(256),
    face_align_eval,                  # face-aligned 224 crop
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])


# -----------------------
# Build datasets (train/eval views)
# -----------------------
# A base dataset (no transform) is useful for consistent target extraction
dataset_base = ImageFolder(
    root=dataset_dir,
    is_valid_file=is_valid_file
)

# Views with different transforms but the same underlying index/targets
dataset_train_view = ImageFolder(
    root=dataset_dir,
    transform=transform_train,
    is_valid_file=is_valid_file
)

dataset_eval_view = ImageFolder(
    root=dataset_dir,
    transform=transform_eval,
    is_valid_file=is_valid_file
)

# -----------------------
# Split choice
# -----------------------
# Set this to True for open-set split (train/test subjects disjoint).
OPEN_SET_BY_CLASS = False
SEED = 42
TEST_RATIO = 0.2  # 70% train, 30% test by default

targets = dataset_base.targets  # consistent with both views
rng = random.Random(SEED)

if not OPEN_SET_BY_CLASS:
    # -------- Closed-set: stratified image-level split per class --------
    indices_by_class = defaultdict(list)
    for idx, y in enumerate(targets):
        indices_by_class[y].append(idx)

    train_indices, test_indices = [], []
    for y, idxs in indices_by_class.items():
        rng.shuffle(idxs)
        n = len(idxs)
        if n == 1:
            # Keep singleton in train to avoid empty test per class
            train_indices.extend(idxs)
            continue
        n_test = max(1, int(round(TEST_RATIO * n)))
        n_train = n - n_test
        # Guard rails to keep both splits non-empty
        if n_train == 0:
            n_train, n_test = 1, n - 1
        train_indices.extend(idxs[:n_train])
        test_indices.extend(idxs[n_train:])

else:
    # -------- Open-set: split at the class (subject) level --------
    all_classes = list(range(len(dataset_base.classes)))
    rng.shuffle(all_classes)
    n_cls = len(all_classes)
    n_test_cls = max(1, int(round(TEST_RATIO * n_cls)))
    test_cls = set(all_classes[:n_test_cls])
    train_cls = set(all_classes[n_test_cls:])

    train_indices = [i for i, y in enumerate(targets) if y in train_cls]
    test_indices  = [i for i, y in enumerate(targets) if y in test_cls]


# -----------------------
# Subsets & loaders
# -----------------------
train_set = Subset(dataset_train_view, train_indices)
test_set  = Subset(dataset_eval_view,  test_indices)

# Adjust num_workers to your environment; pin_memory=True is good for GPU training
train_loader = DataLoader(train_set, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

# Diagnostics
print(f"Subjects (classes): {len(dataset_base.classes)}")
print(f"Class names: {dataset_base.classes}")
print(f"Train/Test sizes: {len(train_set)}, {len(test_set)}")
print(f"Open-set by class: {OPEN_SET_BY_CLASS}")

Subjects (classes): 33
Class names: ['0840a551-9721-4434-964e-7a40a110a0bb', '0991877e-9ec8-4f79-b13a-84e3d9103011', '12c70db2-fbdf-4316-ac6a-482cd23952b8', '14ec96d0-41ef-4cdc-830f-e51ccc199189', '1f04d012-9569-4d97-81e7-2897b19a8605', '20b976ec-f3d3-467c-afff-8fc648b62a0d', '32f9d687-edcf-429e-8fcd-5a6637864cbe', '396f315d-d810-499f-9c61-cd3b447d88fc', '48f54667-8eff-4684-a5c7-034ec4d15fcc', '4b2e1202-1107-4b8d-a2e3-abd0afe94562', '50ec9984-a45f-4d1d-8689-33a6bc49c43f', '60097505-c938-47c4-aec9-2ac9e915c3f1', '6c9462a3-9880-4ffa-b454-66650f61b602', '7ca68cde-2ae1-4c0e-bdd2-7d7901f6a2a2', '7cbef553-8ff3-40a8-9aaa-64ccfc13b673', '856d7434-4ddb-4b06-b13a-a93542c8812b', '872a3aa4-2af4-4b65-b3f0-1bbaaf36d2e4', '8d185d8d-6216-4684-a161-bd564a3645e5', '95d013fa-eb14-42ed-8b6a-d01f51516f8f', '98bda299-9733-420c-a7dc-7449f5f2f2c3', '9cb4ba22-8ec3-4e98-96a6-e7a4a4bed3a1', 'a16a43bc-d800-497e-baf2-83f9ac6ec97b', 'aa1b7628-6e73-4030-9319-ff3b03cba2e2', 'aa7f4402-ff80-4a0d-b70b-40b6f9e4c000', 'ab

**Model development MobineNet-V2**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import mobilenet_v2

# -----------------------
# Model
# -----------------------
model = mobilenet_v2(weights="IMAGENET1K_V1")  # torchvision >= 0.13
model.classifier[1] = nn.Linear(model.last_channel, len(dataset_train_view.classes))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# -----------------------
# Loss, optimizer, scheduler
# -----------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# -----------------------
# Train/Test loop (no validation subset)
# -----------------------
num_epochs = 5  # adjust as needed

for epoch in range(num_epochs):
    # ---- Train ----
    model.train()
    running_loss, running_corrects, n_train = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        preds = outputs.argmax(dim=1)
        running_loss += loss.item() * images.size(0)
        running_corrects += (preds == labels).sum().item()
        n_train += images.size(0)

    train_loss = running_loss / max(1, n_train)
    train_acc = running_corrects / max(1, n_train)

    scheduler.step()

    # ---- Test ----
    model.eval()
    test_loss_sum, test_corrects, n_test = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            preds = outputs.argmax(dim=1)

            test_loss_sum += loss.item() * images.size(0)
            test_corrects += (preds == labels).sum().item()
            n_test += images.size(0)

    test_loss = test_loss_sum / max(1, n_test)
    test_acc = test_corrects / max(1, n_test)

    print(f"Epoch {epoch+1}/{num_epochs} "
          f"| train_loss={train_loss:.4f} acc={train_acc:.3f} "
          f"| test_loss={test_loss:.4f} acc={test_acc:.3f}")

print("Training complete.")

# -----------------------
# Final test evaluation & optional confusion matrix
# -----------------------
model.eval()
num_classes = len(dataset_train_view.classes)
conf_mat = torch.zeros(num_classes, num_classes, dtype=torch.int64)  # [true, pred]
final_loss_sum, final_corrects, final_n = 0.0, 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        preds = outputs.argmax(dim=1)

        final_loss_sum += loss.item() * images.size(0)
        final_corrects += (preds == labels).sum().item()
        final_n += images.size(0)

        # Update confusion matrix
        for t, p in zip(labels.view(-1), preds.view(-1)):
            conf_mat[t.long(), p.long()] += 1

final_test_loss = final_loss_sum / max(1, final_n)
final_test_acc = final_corrects / max(1, final_n)
print(f"Final Test: loss={final_test_loss:.4f} acc={final_test_acc:.3f}")

# Per-class accuracy
per_class_total = conf_mat.sum(dim=1)
per_class_correct = conf_mat.diag()
per_class_acc = torch.where(per_class_total > 0,
                            per_class_correct.float() / per_class_total.float(),
                            torch.zeros_like(per_class_total, dtype=torch.float))

# Print a few per-class accuracies (or all)
class_names = dataset_train_view.classes
for i, acc in enumerate(per_class_acc):
    print(f"Class {i} ({class_names[i]}): acc={acc:.3f}")




Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 156MB/s]


Epoch 1/5 | train_loss=2.8606 acc=0.338 | test_loss=1.8903 acc=0.663
Epoch 2/5 | train_loss=1.3643 acc=0.806 | test_loss=0.8383 acc=0.886
Epoch 3/5 | train_loss=0.6304 acc=0.928 | test_loss=0.3679 acc=0.940
Epoch 4/5 | train_loss=0.3302 acc=0.972 | test_loss=0.2369 acc=0.946
Epoch 5/5 | train_loss=0.1992 acc=0.987 | test_loss=0.1615 acc=0.976
Training complete.
Final Test: loss=0.1615 acc=0.976
Class 0 (0840a551-9721-4434-964e-7a40a110a0bb): acc=1.000
Class 1 (0991877e-9ec8-4f79-b13a-84e3d9103011): acc=1.000
Class 2 (12c70db2-fbdf-4316-ac6a-482cd23952b8): acc=1.000
Class 3 (14ec96d0-41ef-4cdc-830f-e51ccc199189): acc=1.000
Class 4 (1f04d012-9569-4d97-81e7-2897b19a8605): acc=1.000
Class 5 (20b976ec-f3d3-467c-afff-8fc648b62a0d): acc=1.000
Class 6 (32f9d687-edcf-429e-8fcd-5a6637864cbe): acc=1.000
Class 7 (396f315d-d810-499f-9c61-cd3b447d88fc): acc=1.000
Class 8 (48f54667-8eff-4684-a5c7-034ec4d15fcc): acc=0.909
Class 9 (4b2e1202-1107-4b8d-a2e3-abd0afe94562): acc=1.000
Class 10 (50ec9984-a45

**MobileFaceNet**

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ✅ We only need the recognition module (MobileFaceNet) from the pack
from insightface.app import FaceAnalysis

# -----------------------
# Config
# -----------------------
USE_GPU = torch.cuda.is_available()
device = torch.device('cuda' if USE_GPU else 'cpu')
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if USE_GPU else ['CPUExecutionProvider']

# MobileFaceNet in the InsightFace pack:
# - 'buffalo_s' → MobileFaceNet (MBF) @ WebFace600K (good balance of speed/accuracy)
#   If you want an even smaller/compact version, use 'buffalo_sc' (MBF compact).
app = FaceAnalysis(name='buffalo_s', providers=providers, allowed_modules=['recognition'])
app.prepare(ctx_id=0 if USE_GPU else -1)

rec = app.models.get('recognition', None)
assert rec is not None, "Recognition (MobileFaceNet) model not loaded."

# InsightFace MobileFaceNet outputs a normalized embedding vector.
# Most packs output 512-D (some compact variants output 128-D).
EMBED_DIM = getattr(rec, 'output_size', 512)  # fallback to 512 if attribute not found

num_classes = len(dataset_train_view.classes)

# -----------------------
# ArcFace margin head
# -----------------------
class ArcMarginProduct(nn.Module):
    """
    ArcFace head: logits = s * cos(theta + m) for target class, s * cos(theta) otherwise.
    Expects normalized features and learned normalized weights.
    """
    def __init__(self, in_features, out_features, s=64.0, m=0.50, easy_margin=False):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.easy_margin = easy_margin
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input, labels):
        # Normalize features and weights to the hypersphere
        cosine = torch.nn.functional.linear(
            torch.nn.functional.normalize(input),
            torch.nn.functional.normalize(self.weight)
        )
        sine = torch.sqrt(torch.clamp(1.0 - cosine ** 2, min=0.0))
        phi = cosine * self.cos_m - sine * self.sin_m  # cos(theta + m)

        if self.easy_margin:
            phi = torch.where(cosine > 0, phi, cosine)
        else:
            phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)

        logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return logits * self.s

# -----------------------
# "Model": we train only the ArcMargin head; backbone is frozen
# -----------------------
arc_head = ArcMarginProduct(in_features=EMBED_DIM, out_features=num_classes, s=64.0, m=0.50).to(device)

# -----------------------
# Loss, optimizer, scheduler
# -----------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(arc_head.parameters(), lr=1e-3)  # head-only can use a slightly higher LR
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# -----------------------
# Helper: batch embeddings from MobileFaceNet
#   Input: (B,C,H,W) tensors with values in [-1,1]
#   Output: (B, EMBED_DIM)
# -----------------------
@torch.no_grad()
def batch_mobilefacenet_embeddings(images_t: torch.Tensor) -> torch.Tensor:
    embs = []
    for img_t in images_t:
        # Convert back to uint8 HWC for InsightFace runtime
        img = img_t.detach().cpu().permute(1, 2, 0).numpy()           # HWC, float [-1,1]
        img_uint8 = np.clip((img * 0.5 + 0.5) * 255.0, 0, 255).astype(np.uint8)
        feat = rec.get_embedding(img_uint8)                            # (EMBED_DIM,) np.float32, L2-normalized
        embs.append(torch.from_numpy(feat).float())
    return torch.stack(embs, dim=0)                                    # (B, EMBED_DIM)

# -----------------------
# Train/Test loop (no validation subset)
# -----------------------
num_epochs = 5  # adjust as needed

for epoch in range(num_epochs):
    # ---- Train ----
    arc_head.train()
    running_loss, running_corrects, n_train = 0.0, 0, 0

    for images, labels in train_loader:
        labels = labels.to(device)

        # Frozen backbone → extract embeddings
        with torch.no_grad():
            embs = batch_mobilefacenet_embeddings(images)
        embs = embs.to(device)

        optimizer.zero_grad()
        outputs = arc_head(embs, labels)  # margin logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        preds = outputs.argmax(dim=1)
        running_loss += loss.item() * images.size(0)
        running_corrects += (preds == labels).sum().item()
        n_train += images.size(0)

    train_loss = running_loss / max(1, n_train)
    train_acc = running_corrects / max(1, n_train)

    scheduler.step()

    # ---- Test ----
    arc_head.eval()
    test_loss_sum, test_corrects, n_test = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            labels = labels.to(device)
            embs = batch_mobilefacenet_embeddings(images).to(device)
            outputs = arc_head(embs, labels)
            loss = criterion(outputs, labels)
            preds = outputs.argmax(dim=1)

            test_loss_sum += loss.item() * images.size(0)
            test_corrects += (preds == labels).sum().item()
            n_test += images.size(0)

    test_loss = test_loss_sum / max(1, n_test)
    test_acc = test_corrects / max(1, n_test)

    print(f"Epoch {epoch+1}/{num_epochs} "
          f"| train_loss={train_loss:.4f} acc={train_acc:.3f} "
          f"| test_loss={test_loss:.4f} acc={test_acc:.3f}")

print("Training complete.")

# -----------------------
# Final test evaluation & optional confusion matrix
# -----------------------
arc_head.eval()
num_classes = len(dataset_train_view.classes)
conf_mat = torch.zeros(num_classes, num_classes, dtype=torch.int64)  # [true, pred]
final_loss_sum, final_corrects, final_n = 0.0, 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        labels = labels.to(device)
        embs = batch_mobilefacenet_embeddings(images).to(device)
        outputs = arc_head(embs, labels)
        loss = criterion(outputs, labels)
        preds = outputs.argmax(dim=1)

        final_loss_sum += loss.item() * images.size(0)
        final_corrects += (preds == labels).sum().item()
        final_n += images.size(0)

        # Update confusion matrix
        for t, p in zip(labels.view(-1), preds.view(-1)):
            conf_mat[t.long(), p.long()] += 1

final_test_loss = final_loss_sum / max(1, final_n)
final_test_acc = final_corrects / max(1, final_n)
print(f"Final Test: loss={final_test_loss:.4f} acc={final_test_acc:.3f}")

# Per-class accuracy
per_class_total = conf_mat.sum(dim=1)
per_class_correct = conf_mat.diag()
per_class_acc = torch.where(per_class_total > 0,
                            per_class_correct.float() / per_class_total.float(),
                            torch.zeros_like(per_class_total, dtype=torch.float))

class_names = dataset_train_view.classes
for i, acc in enumerate(per_class_acc):
    print(f"Class {i} ({class_names[i]}): acc={acc:.3f}")


download_path: /root/.insightface/models/buffalo_s


100%|██████████| 124617/124617 [00:05<00:00, 21729.46KB/s]
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_s/1k3d68.onnx landmark_3d_68
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_s/2d106det.onnx landmark_2d_106
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_s/det_500m.onnx detection
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_s/genderage.onnx genderage
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_s/w600k_mbf.onnx recognition ['None', 3, 112, 112] 127.5 127.5


AssertionError: 